# 03 — Dataset B Controlled Preprocessing Pilot

Thin Lightning AI Studio wrapper around the reusable `tdmec_pilot` modules. All
logic lives in `src/tdmec_pilot/` and `scripts/run_dataset_b_pilot.py` — this
notebook only supplies local Studio paths, configuration, and runs the pipeline.

**Safety:** processes exactly `statuses-2.xlsx` and `statuses-69.xlsx`; never the
other files; no embeddings; no training; source files are read-only.

**Persistent outputs:** written under `<output_root>/pilot/<run_id>/` on the
Studio filesystem. The pilot is resumable — re-run the last cell with the same
`--run-id` to continue after an interruption.

In [ ]:
# 1) Lightning AI Studio local paths (no Google Drive mount).
from pathlib import Path

STUDIO_ROOT = Path("/teamspace/studios/this_studio")
REPO_ROOT = STUDIO_ROOT / "community-evolution-modeling"
OUTPUT_ROOT = STUDIO_ROOT / "TDMEC_PROJECT_OUTPUTS"
DATASET_B_ROOT = STUDIO_ROOT / "Dataset B" / "statuses_data"
NODE_INDEX_MAP_PATH = OUTPUT_ROOT / "manifests" / "node_index_map.parquet"

assert STUDIO_ROOT.is_dir(), f"STOP: Studio root missing: {STUDIO_ROOT}"
assert REPO_ROOT.is_dir(), f"STOP: Repository missing: {REPO_ROOT}"
assert DATASET_B_ROOT.is_dir(), f"STOP: Dataset B missing: {DATASET_B_ROOT}"
assert NODE_INDEX_MAP_PATH.is_file(), f"STOP: Node map missing: {NODE_INDEX_MAP_PATH}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("STUDIO_ROOT =", STUDIO_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("DATASET_B_ROOT =", DATASET_B_ROOT)
print("NODE_INDEX_MAP_PATH =", NODE_INDEX_MAP_PATH)

In [ ]:
# 2) Install / expose the local repository packages.
import os
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[test]"],
    cwd=REPO_ROOT,
    check=False,
)
src = str(REPO_ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)
print("src on path:", src)

In [ ]:
# 3) Configuration — local Studio Dataset B source and node-index map.
os.environ["DATASET_B_SOURCE"] = os.environ.get(
    "DATASET_B_SOURCE",
    f"local:{DATASET_B_ROOT}",
)
os.environ["PILOT_OUTPUT_ROOT"] = str(OUTPUT_ROOT)
os.environ["NODE_INDEX_MAP_PATH"] = str(NODE_INDEX_MAP_PATH)
CONFIG = str(REPO_ROOT / "configs" / "dataset_b_pilot.yaml")
print("DATASET_B_SOURCE =", os.environ["DATASET_B_SOURCE"])
print("config =", CONFIG)

In [ ]:
# 4) Run the pilot (thin call into the reusable pipeline).
from tdmec_pilot.config import load_pilot_config
from tdmec_pilot.pipeline import PilotPipeline

cfg = load_pilot_config(CONFIG)
pipe = PilotPipeline(
    cfg,
    dataset_b_source=os.environ["DATASET_B_SOURCE"],
    output_root=os.environ["PILOT_OUTPUT_ROOT"],
    node_index_map_path=os.environ["NODE_INDEX_MAP_PATH"],
    cache_root=os.environ.get("DISCOVERY_CACHE_ROOT", "/tmp/tdmec_cache"),
    # run_id='<existing_run_id>',  # <- set this to RESUME an interrupted run
)
report = pipe.run()
print("run_id     :", report["run_id"])
print("all_passed :", report["all_passed"])
print("accounting :", report["accounting"])
report["gates"]

In [ ]:
# 5) Inspect the run outputs.
import json
import pathlib

run_dir = pathlib.Path(pipe.layout.root)
for p in sorted(run_dir.rglob("*")):
    if p.is_file():
        print(p.relative_to(run_dir), p.stat().st_size, "bytes")
print("\nvalidation_report.json:")
print(
    json.dumps(
        json.loads((run_dir / "validation_report.json").read_text())["gates"],
        indent=2,
    )
)

## Resume

If the run is interrupted, re-run cell 4 with `run_id='<the run_id printed above>'`.
Completed, checksum-verified chunks are skipped.